In [ ]:
from utils import *

In [ ]:
def evalModel(config, model, test):
    model.eval()

    metrics = {}

    progress = 0

    with torch.no_grad():
        try:
            for inputs, targets in test:
                history, future = targets.dischargeHistory, targets.dischargeFuture
                thresholds, means, deviations = targets.thresholds, targets.mean.unsqueeze(-1), targets.deviation.unsqueeze(-1)
                hindcast, forecast = model(inputs)

                # Consider sampling?
                medianPrediction = torch.sum(forecast[0] * forecast[3], dim=-1)
                mae = torch.abs(medianPrediction - future)

                nsePred = medianPrediction * deviations + means
                nseObs = future * deviations + means

                nseNum = torch.sum(torch.pow(nseObs - nsePred, 2))
                nseDenom = torch.sum(torch.pow(nseObs - means, 2))

                medianPrediction = medianPrediction.unsqueeze(-1)
                future = future.unsqueeze(-1)
                thresholds = thresholds.unsqueeze(1).expand(-1, config.future, -1)
                tp = (medianPrediction > thresholds).float() * (future > thresholds).float()
                fp = (medianPrediction > thresholds).float() * (future < thresholds).float()
                fn = (medianPrediction < thresholds).float() * (future > thresholds).float()

                past, _ = inputs
                for n, name in enumerate(past.grdcID):
                    if name not in metrics:
                        metrics[name] = {
                            "iter": 0,
                            "mae": torch.zeros([config.future]),
                            "tp": torch.zeros([config.future, thresholds.shape[-1]]),
                            "fp": torch.zeros([config.future, thresholds.shape[-1]]),
                            "fn": torch.zeros([config.future, thresholds.shape[-1]]),
                            "nodes": past.nodes[n],
                            "nseNum": 0,
                            "nseDenom": 0,
                            "size": None
                        }

                    metrics[name]["mae"] = (metrics[name]["mae"] * metrics[name]["iter"] + mae[n]) / (metrics[name]["iter"] + 1)
                    metrics[name]["tp"] += tp[n]
                    metrics[name]["fp"] += fp[n]
                    metrics[name]["fn"] += fn[n]
                    metrics[name]["nseNum"] += nseNum
                    metrics[name]["nseDenom"] += nseDenom
                    metrics[name]["iter"] += 1

                progress += 1
                print(f"\r{progress}/{len(test)} | {(progress / len(test)) * 100:.2f}% Complete", end="")
        except KeyboardInterrupt:
            pass

    nodeX, nodeY = np.zeros([len(metrics), 4]), np.zeros([len(metrics), 4])
    precisionBox, recallBox, f1Box = np.zeros([len(metrics), config.future, 4]), np.zeros([len(metrics), config.future, 4]), np.zeros([len(metrics), config.future, 4])
    for i, name in enumerate(metrics):
        nodeX[i, :] = metrics[name]["nodes"].cpu().numpy()

        tp = metrics[name]["tp"]
        fp = metrics[name]["fp"]
        fn = metrics[name]["fn"]

        recall = tp / (tp + fn)
        precision = tp / (tp + fp)

        f1 = 2 * recall * precision / (recall + precision)

        nodeY[i, :] = torch.mean(f1, dim=0).cpu().numpy()

        precisionBox[i] = precision.cpu().numpy()
        recallBox[i] = recall.cpu().numpy()
        f1Box[i] = f1.cpu().numpy()

    plt.figure(figsize=(20, 12))

    labels = ["1 Year Return Period", "2 Year Return Period", "5 Year Return Period", "10 Year Return Period"]
    colors = ["blue", "green", "yellow", "orange"]
    for i in range(4):
        plt.subplot(2, 4, i + 1)
        plt.title(labels[i])
        currentX = nodeX[:, i]
        currentY = nodeY[:, i]
        mask = ~np.isnan(currentY)
        currentX, currentY = currentX[mask], currentY[mask]
        plt.scatter(currentX, currentY, alpha=0.3, c=colors[i])

        plt.grid()
        plt.xlabel("Total Upstream Basin Nodes")
        plt.ylabel("F1 Score")

    for i in range(4):
        plt.subplot(2, 4, i + 5)
        plt.title(labels[i])
        currentF1 = f1Box[:, :, i].T
        currentF1 = [box[~np.isnan(box)] for box in currentF1]
        plt.boxplot(currentF1)

        plt.grid()
        plt.xlabel("Forecast Horizon")
        plt.ylabel("F1 Score")

    plt.show()

    # [basins, 1], [basins, timesteps]
    nodeErrorX, nodeErrorY = np.array([metrics[name]["nodes"].cpu().numpy() for name in metrics]), np.array([metrics[name]["mae"].cpu().numpy() for name in metrics])
    plt.figure(figsize=(20, 6))
    for i in range(config.future):
        plt.subplot(1, config.future, i + 1)
        plt.scatter(nodeErrorX, nodeErrorY[:, i])
        plt.title(f"NMAE by Graph Size @ Horizon {i + 1}")
        plt.xlabel("Graph Size (# of Nodes)")
        if i == 0:
            plt.ylabel("NMAE")
        plt.grid()
    plt.show()
    
    nse = 1 - np.array([metrics[name]["nseNum"].cpu().numpy() / metrics[name]["nseDenom"].cpu().numpy() for name in metrics])
    cdf = np.array([np.sum(nse < (threshold / 1000)) / len(nse)] for threshold in range(1000))

    plt.figure(figsize=(12, 6))
    plt.plot(np.arange(1000), cdf)
    plt.title("Cumulative Distribution of NSE")
    plt.xlabel("NSE")
    plt.ylabel("CDF")
    plt.grid()
    plt.show()

    # TODO: Evaluate on more stratified data (other continents?)
    # TODO: Maybe evaluate at New Madrid for fun

In [ ]:
modelPath = os.path.join("checkpoints", "")
config = Config().load(os.path.join(modelPath, "config.json"))
model = InundationGCLSTMStation(config)
model.load_state_dict(os.path.join(modelPath, "checkpoint.pt"), weights_only=True)
dataset = InundationData(config)
train, test = InundationData.split(dataset, trainSplit=config.dataSplit, seed=config.seed)

In [ ]:
evalModel(config, model, test)